# PyTorch — Chapter 6: Multilayer Perceptron Building Blocks in PyTorch


## 1. Xây model bằng nn.Sequential

3 cách viết cùng một model MLP:

- **Một khối duy nhất**: liệt kê layer trực tiếp trong `nn.Sequential(...)`, xử lý theo đúng thứ tự khai báo.
- **OrderedDict**: gán tên riêng cho từng layer (`'dense1'`, `'act1'`,...) — dùng khi cần truy cập layer theo tên thay vì index.
- **Thêm từng layer bằng `add_module()`**: xây model theo điều kiện, không cần liệt kê hết trong một lệnh.

## 2. Model input

- VD `nn.Linear(764, 100)` — input 764 chiều, output 100 chiều. Kích thước batch luôn ẩn: đưa vào tensor `(n, 764)` thì nhận về `(n, 100)`.

## 3. Layer, activation và thuộc tính layer

- Layer phổ biến: `nn.Linear` (fully-connected), `nn.Conv2d` (ảnh), `nn.Dropout` (regularization), `nn.Flatten` (làm phẳng).
- Activation phổ biến: `nn.ReLU` (dùng nhiều nhất hiện nay), `nn.Sigmoid`/`nn.Tanh` (phổ biến trong tài liệu cũ), `nn.Softmax` (chuyển vector thành giá trị dạng xác suất, dùng cho classification).
- Hầu hết layer nhận thêm 2 tham số tuỳ chọn: `device` (chạy trên `"cpu"` hay `"cuda:0"`) và `dtype` (kiểu dữ liệu, mặc định float32).

## 4. Loss function và optimizer

- Loss đo khoảng cách giữa output model và nhãn thật (ground truth): `nn.MSELoss()` (hồi quy), `nn.CrossEntropyLoss()` (phân loại đa lớp), `nn.BCELoss()` (phân loại nhị phân).
- `loss = loss_fn(output, label)` trả về một tensor hỗ trợ autograd — gọi `loss.backward()` để tính gradient.
- Optimizer cần được cấp danh sách tham số cần tối ưu: `torch.optim.Adam(model.parameters(), lr=0.001)`. Các lựa chọn phổ biến: `Adam`, `NAdam` (Adam + Nesterov momentum), `SGD`, `RMSprop`.

## 5. Training và inference

- PyTorch không có hàm `fit()` dựng sẵn — training loop tối giản là `for epoch: y_pred=model(X) → loss → zero_grad() → backward() → step()` (đúng khung xương đã thấy xuyên suốt Chapter 1 và Chapter 4).
- Inference chỉ là gọi `model(X)` trực tiếp lấy `y_pred`. Model luôn kỳ vọng input là **batch**.

## 6. Kiểm tra và lưu/tải model

- `print(model)` in cấu trúc từng layer theo thứ tự.
- Lưu **toàn bộ object** model: `torch.save(model, "my_model.pth")` → tải lại bằng `torch.load(...)`.
- Cách khuyến nghị: chỉ lưu **trọng số** (`model.state_dict()`), nhưng khi tải lại phải tự dựng lại đúng kiến trúc model trước rồi mới `load_state_dict()`.


## 7. Vận dụng


**6.2** — Xây MLP bằng nn.Sequential (liệt kê trực tiếp)

In [1]:
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(764, 100),
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, 10),
    nn.Sigmoid()
)
print(model)


Sequential(
  (0): Linear(in_features=764, out_features=100, bias=True)
  (1): ReLU()
  (2): Linear(in_features=100, out_features=50, bias=True)
  (3): ReLU()
  (4): Linear(in_features=50, out_features=10, bias=True)
  (5): Sigmoid()
)


**6.3** — Xây MLP bằng nn.Sequential + OrderedDict (đặt tên layer)

In [2]:
from collections import OrderedDict
import torch.nn as nn

model = nn.Sequential(OrderedDict([
    ('dense1', nn.Linear(764, 100)),
    ('act1', nn.ReLU()),
    ('dense2', nn.Linear(100, 50)),
    ('act2', nn.ReLU()),
    ('output', nn.Linear(50, 10)),
    ('outact', nn.Sigmoid()),
]))
print(model)


Sequential(
  (dense1): Linear(in_features=764, out_features=100, bias=True)
  (act1): ReLU()
  (dense2): Linear(in_features=100, out_features=50, bias=True)
  (act2): ReLU()
  (output): Linear(in_features=50, out_features=10, bias=True)
  (outact): Sigmoid()
)


**6.4** — Xây MLP bằng cách thêm từng layer với add_module()

In [3]:
import torch.nn as nn

model = nn.Sequential()
model.add_module("dense1", nn.Linear(8, 12))
model.add_module("act1", nn.ReLU())
model.add_module("dense2", nn.Linear(12, 8))
model.add_module("act2", nn.ReLU())
model.add_module("output", nn.Linear(8, 1))
model.add_module("outact", nn.Sigmoid())
print(model)


Sequential(
  (dense1): Linear(in_features=8, out_features=12, bias=True)
  (act1): ReLU()
  (dense2): Linear(in_features=12, out_features=8, bias=True)
  (act2): ReLU()
  (output): Linear(in_features=8, out_features=1, bias=True)
  (outact): Sigmoid()
)


**6.12** — Lưu toàn bộ model bằng torch.save()

In [4]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(764, 100),
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, 10),
    nn.Sigmoid()
)
torch.save(model, "my_model.pth")


**6.13** — Tải lại toàn bộ model bằng torch.load()

In [5]:
import torch

model = torch.load("my_model.pth")


/tmp/ipykernel_11611/3077693230.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model = torch.load("my_model.pth")


**6.14** — Lưu trọng số model bằng state_dict()

In [6]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(764, 100),
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, 10),
    nn.Sigmoid()
)
torch.save(model.state_dict(), "my_model.pth")


**6.15** — Tải lại trọng số bằng load_state_dict()

In [8]:
import torch
import torch.nn as nn

model = nn.Sequential(
    nn.Linear(764, 100),
    nn.ReLU(),
    nn.Linear(100, 50),
    nn.ReLU(),
    nn.Linear(50, 10),
    nn.Sigmoid()
)
model.load_state_dict(torch.load("my_model.pth", weights_only=True))


<All keys matched successfully>